# Exploding Kittens — Hand-Test

`BACKEND` + `NUM_PLAYERS` setzen → Setup-Zellen → `make_turn_ui()`.

Spielerzahl: **Claude & GLM 2–5** (Regelwerk). Default **3** für Zwei-Menschen-Tests (P0+P1 menschlich, Rest Bot).

Notizen: `meeting/10.7/THOUGHTS.txt` · Kernel neu starten nach Backend-Wechsel.

In [40]:
import importlib.util
import random
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve()
OUTPUT_DIR = REPO_ROOT / "outputs"

BACKEND = "codex"  # gpt | codex | claude | glm
NUM_PLAYERS = 3
START_PLAYER = 0
SEED = 42
HUMAN_PLAYERS = {0, 1}

AUTO_VIEW = True
SHOW_CHEAT = False
AUTO_SETUP = True
AUTO_BOT_PASS = True

BACKENDS = {
    "gpt": "expl_gpt_ag.py",
    "codex": "expl_codex_ag.py",
    "claude": "expl_claude_ag.py",
    "glm": "expl_glm_ag.py",
}

CODE_PATH = OUTPUT_DIR / BACKENDS[BACKEND]
if not CODE_PATH.exists():
    raise FileNotFoundError(f"Missing {CODE_PATH}")

print(f"backend={BACKEND}  players={NUM_PLAYERS}  seed={SEED}  humans={sorted(HUMAN_PLAYERS)}")

backend=codex  players=3  seed=42  humans=[0, 1]


In [41]:
from IPython.display import clear_output, display

TERMINAL, CHANCE, SIMULTANEOUS = -1, -2, -3
PLAYER_LABELS = {TERMINAL: "TERMINAL", CHANCE: "CHANCE", SIMULTANEOUS: "SIMULTANEOUS"}


def load_game_module(code_path: Path):
    name = f"manual_test_{code_path.stem}"
    spec = importlib.util.spec_from_file_location(name, code_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def make_game(Game, *, num_players: int, start_player: int = 0, seed: int = 42):
    if BACKEND == "glm":
        try:
            return Game(num_players=num_players, seed=seed)
        except TypeError:
            return Game(num_players)
    for kwargs in (
        {"num_players": num_players, "start_player": start_player},
        {"num_players": num_players, "seed": seed},
        {"num_players": num_players},
    ):
        try:
            return Game(**kwargs)
        except TypeError:
            continue
    return Game(num_players)


def player_label(game, state) -> str:
    cp = game.current_player(state)
    return PLAYER_LABELS.get(cp, f"p{cp}")


def _pick_chance(game, state, rng: random.Random):
    if hasattr(game, "chance_outcomes"):
        out = game.chance_outcomes(state)
        if out:
            acts, w = zip(*out)
            return rng.choices(list(acts), weights=list(w), k=1)[0]
    return rng.choice(game.legal_actions(state))


def auto_resolve_bots(game, state, *, rng: random.Random, max_steps: int = 500):
    steps = 0
    while steps < max_steps and not game.is_terminal(state):
        phase = getattr(state, "phase", None)
        cp = game.current_player(state)

        if AUTO_SETUP and BACKEND == "claude" and phase in ("DEAL", "BUILD"):
            state = game.apply_action(state, _pick_chance(game, state, rng))
            steps += 1
            continue
        if AUTO_SETUP and BACKEND == "codex" and phase in ("setup_deal", "setup_deck"):
            state = game.apply_action(state, _pick_chance(game, state, rng))
            steps += 1
            continue

        if not AUTO_BOT_PASS or not HUMAN_PLAYERS:
            break

        if cp in HUMAN_PLAYERS:
            break

        if BACKEND == "claude" and phase == "NOPE":
            state = game.apply_action(state, "pass")
        elif BACKEND == "codex" and phase == "nope":
            state = game.apply_action(state, "nope:pass")
        elif BACKEND == "glm" and phase == "nope_window":
            state = game.apply_action(state, ("nope_pass",))
        elif BACKEND == "glm" and phase == "favor_give":
            state = game.apply_action(state, game.legal_actions(state)[0])
        elif cp == CHANCE:
            if BACKEND == "claude" and phase == "STEAL":
                state = game.apply_action(state, _pick_chance(game, state, rng))
            elif BACKEND == "codex" and phase == "steal_random":
                state = game.apply_action(state, _pick_chance(game, state, rng))
            else:
                break
        else:
            break
        steps += 1
    return state, steps


def resolve_action(game, action, legal):
    if action in legal:
        return action
    if isinstance(action, str):
        resolved = game.name_to_action(action)
        if resolved in legal:
            return resolved
    raise ValueError(f"Illegal action {action!r}; legal={legal[:6]}{'...' if len(legal) > 6 else ''}")


def view_for_state(game, state, *, manual_view=None):
    if manual_view is not None:
        return manual_view
    if AUTO_VIEW:
        cp = game.current_player(state)
        if cp >= 0:
            return cp
    return 0


def extra_player_log(state, vp: int) -> None:
    """GLM: see_future etc. nur in log, nicht in information_state."""
    if BACKEND != "glm" or not getattr(state, "log", None):
        return
    tag = f"P{vp}"
    hits = [ln for ln in state.log if tag in ln][-4:]
    if hits:
        print("--- dein Spielverlauf (GLM log) ---")
        for ln in hits:
            print(f"  {ln}")


def play_hint(game, state) -> None:
    phase = getattr(state, "phase", "?")
    if BACKEND == "claude" and phase == "PLAY":
        print(">>> draw = ziehen & Zug beenden | play:attack sofort Wechsel")
    elif BACKEND == "glm" and phase == "main":
        print(">>> pass = ziehen & Zug beenden | play:see_future / play:attack …")
    elif BACKEND == "glm" and phase == "nope_window":
        print(">>> nope_pass oder nope")
    elif BACKEND == "glm" and phase == "favor_give":
        print(">>> favor_give:<karte>")
    elif BACKEND == "glm" and phase == "defuse_place":
        print(">>> defuse_place:<0..decklen>")


def show(game, state, *, view_player=None, full=None) -> None:
    vp = view_for_state(game, state, manual_view=view_player)
    cheat = SHOW_CHEAT if full is None else full
    print(f"current={player_label(game, state)}  phase={getattr(state, 'phase', '?')}  terminal={game.is_terminal(state)}")
    if game.is_terminal(state):
        print("returns:", game.returns(state))
    cp = game.current_player(state)
    if cp in HUMAN_PLAYERS or not HUMAN_PLAYERS:
        print(f">>> Spieler {cp} am Zug")
    elif cp >= 0:
        print(f">>> Bot P{cp} (AUTO_BOT_PASS)")
    print()
    print(f"--- Sicht P{vp} ---")
    print(game.information_state(state, vp))
    extra_player_log(state, vp)
    if cheat:
        print()
        print(game.render(state))
    legal = game.legal_actions(state)
    print()
    play_hint(game, state)
    print(f"--- legal actions ({len(legal)}) ---")
    for i, act in enumerate(legal):
        print(f"  [{i:3d}] {act}  ({game.action_to_name(act)!r})")


def play(game, state, action):
    legal = game.legal_actions(state)
    action = resolve_action(game, action, legal)
    state = game.apply_action(state, action)
    state, _ = auto_resolve_bots(game, state, rng=random.Random(SEED))
    return state


def play_index(game, state, index: int):
    legal = game.legal_actions(state)
    if not 0 <= index < len(legal):
        raise IndexError(f"index {index} out of range 0..{len(legal) - 1}")
    return play(game, state, legal[index])


def new_game(*, seed=None):
    use_seed = seed if seed is not None else SEED
    g = make_game(Game, num_players=NUM_PLAYERS, start_player=START_PLAYER, seed=use_seed)
    state = g.initial_state()
    state, steps = auto_resolve_bots(g, state, rng=random.Random(use_seed))
    if steps:
        print(f"auto: {steps} Bot/Setup-Züge")
    return g, state


def make_turn_ui(game, state, *, manual_view=None):
    import ipywidgets as widgets

    holder = {"game": game, "state": state, "fixed_view": manual_view}
    move_box = widgets.Text(description="Zug:", placeholder="Index oder play:attack / pass …")
    go_btn = widgets.Button(description="Spielen", button_style="primary")
    reset_btn = widgets.Button(description="Neu")
    out = widgets.Output()

    def refresh():
        with out:
            clear_output(wait=True)
            g, s = holder["game"], holder["state"]
            show(g, s, view_player=holder["fixed_view"], full=g.is_terminal(s))

    def submit(_=None):
        raw = move_box.value.strip()
        move_box.value = ""
        if not raw:
            refresh()
            return
        if raw.lower() in {"q", "quit"}:
            return
        if raw.lower() == "debug":
            with out:
                clear_output(wait=True)
                show(holder["game"], holder["state"], view_player=holder["fixed_view"], full=True)
            return
        try:
            if raw.isdigit():
                holder["state"] = play_index(holder["game"], holder["state"], int(raw))
            else:
                holder["state"] = play(holder["game"], holder["state"], raw)
        except Exception as exc:
            with out:
                clear_output(wait=True)
                print(f"FEHLER: {exc}")
                show(holder["game"], holder["state"], view_player=holder["fixed_view"])
            return
        refresh()

    def reset(_=None):
        g, s = new_game()
        holder["game"], holder["state"] = g, s
        refresh()

    go_btn.on_click(submit)
    reset_btn.on_click(reset)
    if hasattr(move_box, "on_submit"):
        move_box.on_submit(submit)
    display(widgets.VBox([widgets.HBox([move_box, go_btn, reset_btn]), out]))
    refresh()
    return holder

In [35]:
module = load_game_module(CODE_PATH)
Game = module.Game
game, state = new_game()
show(game, state)
if BACKEND == "gpt":
    print("\nWARN: gpt agentic crasht oft bei apply_action (deepcopy bug)")

current=p0  phase=main  terminal=False
>>> Spieler 0 am Zug

--- Sicht P0 ---
Phase: main
You: P0 (ALIVE)
Your turn (turns remaining: 1)
Hand(8): cat_4 cat_5 cat_5 defuse nope see_future skip skip
P1[ALIVE](8 cards)
P2[ALIVE](8 cards)
Deck: 30 cards
Discard: (empty)

>>> pass = ziehen & Zug beenden | play:see_future / play:attack …
--- legal actions (39) ---
  [  0] ('pass',)  ('pass')
  [  1] ('play', 'cat_4')  ('play:cat_4')
  [  2] ('play', 'cat_5')  ('play:cat_5')
  [  3] ('play', 'see_future')  ('play:see_future')
  [  4] ('play', 'skip')  ('play:skip')
  [  5] ('pair', 'cat_5', 1)  ('pair:cat_5:p1')
  [  6] ('pair', 'cat_5', 2)  ('pair:cat_5:p2')
  [  7] ('pair', 'skip', 1)  ('pair:skip:p1')
  [  8] ('pair', 'skip', 2)  ('pair:skip:p2')
  [  9] ('five', ('cat_4', 'cat_5', 'defuse', 'nope', 'see_future'), 'cat_4')  ('five:cat_4:cat_5:defuse:nope:see_future:cat_4')
  [ 10] ('five', ('cat_4', 'cat_5', 'defuse', 'nope', 'see_future'), 'cat_5')  ('five:cat_4:cat_5:defuse:nope:see_futu

In [42]:
turn_ui = make_turn_ui(game, state)

C:\Users\benti\AppData\Local\Temp\ipykernel_31380\2000722569.py:236: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  move_box.on_submit(submit)
